# Active versus Passive Ownership: Extension of the Asset-Embedding Replication

This notebook extends the completed 80-quarter replication. It asks whether the information in institutional holdings—and the performance of holdings-based asset embeddings—differs with the activeness of the investors supplying those holdings.

The completed replication artifacts are treated as immutable inputs. New data and outputs are checkpointed under `active_passive_extension/`; this notebook does not rebuild the RS or Word2Vec models.

## 1. Empirical design

S12 mutual funds are the primary classification universe because `fundno` identifies individual funds. S34 institutions remain useful as a manager-level robustness sample, but S34 `typecode` must not be interpreted as an active/passive flag without a verified codebook.

We will construct three independent signals:

1. **Reported status:** a verified CRSP Mutual Fund index-fund indicator or comparable WRDS field, linked point-in-time to S12 funds.
2. **Active Share:** $AS_{it}=\tfrac12\sum_a|w_{iat}-w^{mkt}_{at}|$, where the market portfolio uses CRSP firm market capitalizations and both portfolios are normalized over a common eligible universe.
3. **Return-adjusted turnover:** $TO_{it}=\tfrac12\sum_a|w_{iat}-w^{BH}_{ia,t}|$, where $w^{BH}$ carries the preceding portfolio forward using intervening asset returns. This separates trading from passive price drift.

The *Green Tilts* paper motivates a further decomposition into an **extensive margin** (entries, exits, inclusions, and exclusions) and an **intensive margin** (reweighting securities that remain held). It does not itself provide an active/passive classification, so our decomposition is inspired by that framework rather than presented as its exact estimator.

Primary comparisons will use each continuous measure and quarterly quintiles. A conservative consensus label will define passive funds as externally indexed with low Active Share and low turnover, active funds as externally active with high Active Share or high turnover, and retain conflicting observations as mixed. Cutoffs will be fixed only after inspecting coverage and stability.

## 2. Environment and output paths

This cell locates the project, imports lightweight dependencies, defines the 80-quarter panel, and creates extension-only storage folders.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings':
    candidate = PROJECT_ROOT / 'Projects' / 'Asset_Embeddings'
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings':
    raise RuntimeError('Open this notebook from the Asset_Embeddings project directory.')
os.chdir(PROJECT_ROOT)

LOCAL_PACKAGES = PROJECT_ROOT / '.packages'
if str(LOCAL_PACKAGES) not in sys.path:
    sys.path.insert(0, str(LOCAL_PACKAGES))

QUARTERS = [(y, q) for y in range(2005, 2025) for q in range(1, 5)]
REPLICATION_ROOT = PROJECT_ROOT / 'full_replication_80q'
EXTENSION_ROOT = PROJECT_ROOT / 'active_passive_extension'
for folder in ['catalog', 'labels', 'measures', 'samples', 'benchmarks', 'figures', 'manifests']:
    (EXTENSION_ROOT / folder).mkdir(parents=True, exist_ok=True)

def qlabel(year, quarter):
    return f'{year}q{quarter}'

def save_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    safe = frame.copy()
    for column in safe.select_dtypes(include='object'):
        safe[column] = safe[column].astype('string')
    safe.to_parquet(path, index=False)

print(f'Project root: {PROJECT_ROOT}')
print(f'Extension output root: {EXTENSION_ROOT}')
print(f'Quarters: {len(QUARTERS)} ({QUARTERS[0]} through {QUARTERS[-1]})')

## 3. Audit reusable local inputs

We need the S12 firm-level holdings before the combined S12–S34 filter, the saved firm characteristic/market-equity panel, and the completed replication sample for downstream comparisons. This audit is read-only and does not download or rebuild anything.

In [ ]:
audit_rows = []
for year, quarter in QUARTERS:
    label = qlabel(year, quarter)
    paths = {
        's12_holdings': REPLICATION_ROOT / 'processed' / label / 's12_firm_holdings_prefilter.parquet',
        'jkp_firms': REPLICATION_ROOT / 'processed' / label / 'jkp_firm_panel.parquet',
        'combined_holdings': REPLICATION_ROOT / 'processed' / label / 'combined_final_holdings.parquet',
        'valuation_split': REPLICATION_ROOT / 'benchmarks' / label / 'evaluation_sample_and_split.parquet',
    }
    row = {'year': year, 'quarter': quarter, 'label': label}
    row.update({name: path.exists() for name, path in paths.items()})
    row['complete'] = all(row[name] for name in paths)
    audit_rows.append(row)

input_audit = pd.DataFrame(audit_rows)
save_parquet(input_audit, EXTENSION_ROOT / 'manifests' / 'local_input_audit.parquet')
display(input_audit.groupby('complete').size().rename('quarters').reset_index())
if not input_audit['complete'].all():
    display(input_audit.loc[~input_audit['complete']])
    print('Missing files are shown above. Do not rebuild yet; we will resolve the exact path first.')
else:
    print('All 80 quarters of immutable replication inputs are available locally.')

## 4. Inspect the local fund panel

This confirms identifiers and fields before any classification logic is written. It also reports how many S12 funds survive the original portfolio-quality filter in each quarter.

In [ ]:
panel_rows = []
for year, quarter in QUARTERS:
    label = qlabel(year, quarter)
    path = REPLICATION_ROOT / 'processed' / label / 's12_firm_holdings_prefilter.parquet'
    if not path.exists():
        continue
    frame = pd.read_parquet(path)
    required = {'investor_id', 'investor_key', 'permco', 'dollar_holding'}
    missing = sorted(required.difference(frame.columns))
    panel_rows.append({
        'year': year, 'quarter': quarter, 'rows': len(frame),
        'funds': frame['investor_id'].nunique() if 'investor_id' in frame else np.nan,
        'firms': frame['permco'].nunique() if 'permco' in frame else np.nan,
        'missing_required_columns': ', '.join(missing),
    })

s12_inventory = pd.DataFrame(panel_rows)
save_parquet(s12_inventory, EXTENSION_ROOT / 'manifests' / 's12_local_inventory.parquet')
display(s12_inventory.describe(include='all'))
sample_path = REPLICATION_ROOT / 'processed' / qlabel(2024, 4) / 's12_firm_holdings_prefilter.parquet'
sample = pd.read_parquet(sample_path)
print('S12 columns:', sample.columns.tolist())
display(sample.head())

## 5. Connect to WRDS and discover reported-label sources

Run this only when WRDS access is available. It searches accessible metadata for mutual-fund index/passive/active fields. We deliberately discover and verify the source before writing a download query; no guessed S34 classification is used. Enter credentials through the normal WRDS prompt—never paste a password into the notebook.

In [ ]:
import wrds

try:
    db.connection.close()
except Exception:
    pass
db = wrds.Connection()
print('WRDS connection is active.')

label_catalog_query = """
select table_schema as library, table_name, ordinal_position,
       column_name, data_type
from information_schema.columns
where (
    position('index' in lower(column_name)) > 0
    or position('passive' in lower(column_name)) > 0
    or position('active' in lower(column_name)) > 0
    or position('objective' in lower(column_name)) > 0
    or lower(column_name) in ('fundno','crsp_fundno','wficn','ticker','fund_name','fundname')
)
and (
    position('fund' in lower(table_schema)) > 0
    or position('fund' in lower(table_name)) > 0
    or table_schema in ('crsp_q_mutualfunds','crsp_a_mutualfunds','tr_mutualfunds')
)
order by table_schema, table_name, ordinal_position
"""
reported_label_catalog = db.raw_sql(label_catalog_query)
save_parquet(reported_label_catalog, EXTENSION_ROOT / 'catalog' / 'reported_label_candidates.parquet')
display(reported_label_catalog)
print(f'Candidate columns found: {len(reported_label_catalog):,}')

## 6. Decision gate before construction

After Sections 2–5 run, inspect the catalog rather than selecting a convenient-looking field automatically. The next notebook stage will: (i) verify the label definition and point-in-time coverage; (ii) create the S12-to-label crosswalk; (iii) construct Active Share using a common CRSP universe; (iv) obtain quarterly firm returns and construct return-adjusted turnover plus extensive/intensive components; and (v) compare RS and Word2Vec performance across continuous activeness and consensus groups.

The design will preserve continuous scores as the primary evidence. Quintiles and consensus labels are presentation devices, not substitutes for the underlying measures.

## 7. Verify the CRSP label and historical dating fields

The catalog identifies `crsp_q_mutualfunds.fund_hdr_hist` as the preferred source because it contains `index_fund_flag` and is historical. This cell retrieves its complete schema, a small sample, and the flag distribution. We must identify the effective-date fields before performing a point-in-time join.

In [ ]:
historical_header_schema = db.raw_sql("""
select ordinal_position, column_name, data_type
from information_schema.columns
where table_schema = 'crsp_q_mutualfunds'
  and table_name = 'fund_hdr_hist'
order by ordinal_position
""")
save_parquet(historical_header_schema, EXTENSION_ROOT / 'catalog' / 'crsp_fund_hdr_hist_schema.parquet')
display(historical_header_schema)

historical_header_sample = db.raw_sql(
    'select * from crsp_q_mutualfunds.fund_hdr_hist limit 10'
)
save_parquet(historical_header_sample, EXTENSION_ROOT / 'catalog' / 'crsp_fund_hdr_hist_sample.parquet')
display(historical_header_sample)

index_flag_distribution = db.raw_sql("""
select coalesce(index_fund_flag, '<NULL>') as index_fund_flag, count(*) as rows
from crsp_q_mutualfunds.fund_hdr_hist
group by index_fund_flag
order by rows desc
""")
save_parquet(index_flag_distribution, EXTENSION_ROOT / 'catalog' / 'crsp_index_flag_distribution.parquet')
display(index_flag_distribution)

## 8. Download the point-in-time S12 fund-ticker bridge

S12’s portfolio-security `ticker` is not a fund identifier. The correct bridge is `tr_mutualfunds.s12type8.fticker`, linked to `fundno` and dated by `fdate`. The extract is small, checkpointed locally, and restricted to the project period.

In [ ]:
s12_ticker_path = EXTENSION_ROOT / 'labels' / 's12_fund_tickers_2005_2024.parquet'
if s12_ticker_path.exists():
    s12_fund_tickers = pd.read_parquet(s12_ticker_path)
    ticker_status = 'checkpoint'
else:
    s12_fund_tickers = db.raw_sql("""
        select fdate, fundno, fticker, tfsdid
        from tr_mutualfunds.s12type8
        where fdate between '2005-01-01' and '2024-12-31'
    """, date_cols=['fdate'])
    save_parquet(s12_fund_tickers, s12_ticker_path)
    ticker_status = 'downloaded'

s12_fund_tickers['fundno'] = pd.to_numeric(s12_fund_tickers['fundno'], errors='coerce')
s12_fund_tickers['fticker_clean'] = (
    s12_fund_tickers['fticker'].astype('string').str.upper().str.strip()
)
s12_fund_tickers = s12_fund_tickers.dropna(subset=['fdate', 'fundno', 'fticker_clean']).copy()
s12_fund_tickers['fundno'] = s12_fund_tickers['fundno'].astype('int64')

ticker_audit = pd.DataFrame({
    'status': [ticker_status],
    'rows': [len(s12_fund_tickers)],
    'funds': [s12_fund_tickers['fundno'].nunique()],
    'tickers': [s12_fund_tickers['fticker_clean'].nunique()],
    'first_fdate': [s12_fund_tickers['fdate'].min()],
    'last_fdate': [s12_fund_tickers['fdate'].max()],
    'fund_date_duplicate_rows': [int(s12_fund_tickers.duplicated(['fdate','fundno'], keep=False).sum())],
    'ticker_date_ambiguous_rows': [int(s12_fund_tickers.duplicated(['fdate','fticker_clean'], keep=False).sum())],
})
save_parquet(ticker_audit, EXTENSION_ROOT / 'manifests' / 's12_ticker_audit.parquet')
display(ticker_audit)
display(s12_fund_tickers.head())

## 9. Next linkage rule

Once Section 7 confirms the historical effective-date columns and the meaning of every flag value, we will join each S12 fund-quarter to the CRSP record valid on that quarter-end. Ticker reuse and many-to-many matches will be rejected or resolved using dated names—not silently retained. Reported status will remain missing where no defensible match exists.

## 10. Discover the CRSP–Thomson MFLINKS bridge

The clean S12 ticker bridge ends in 2018, so ticker matching alone would create severe late-sample coverage loss. This cell searches the accessible MFLINKS libraries and saves their schemas and small samples. It does not yet assume that a similarly named identifier equals S12 `fundno`.

In [ ]:
mflinks_schema = db.raw_sql("""
select table_schema as library, table_name, ordinal_position, column_name, data_type
from information_schema.columns
where position('mflink' in lower(table_schema)) > 0
   or position('mflink' in lower(table_name)) > 0
order by table_schema, table_name, ordinal_position
""")
save_parquet(mflinks_schema, EXTENSION_ROOT / 'catalog' / 'mflinks_schema.parquet')
display(mflinks_schema)

mflinks_tables = mflinks_schema[['library','table_name']].drop_duplicates()
sample_rows = []
for row in mflinks_tables.itertuples(index=False):
    try:
        sample_frame = db.raw_sql(f'select * from {row.library}.{row.table_name} limit 5')
        sample_frame.insert(0, '_table', f'{row.library}.{row.table_name}')
        sample_rows.append(sample_frame)
    except Exception as error:
        print(f'Could not sample {row.library}.{row.table_name}: {error}')

if sample_rows:
    mflinks_samples = pd.concat(sample_rows, ignore_index=True, sort=False)
    save_parquet(mflinks_samples, EXTENSION_ROOT / 'catalog' / 'mflinks_samples.parquet')
    display(mflinks_samples)
else:
    print('No accessible MFLINKS table samples were returned.')

## 11. Construct market-relative Active Share

This construction uses the S12 funds that entered the completed combined replication sample, ensuring that subsequent embedding comparisons use the same investable firms and portfolio-quality filters. The benchmark is the broad US CRSP common-equity universe available in the locally saved JKP/CRSP extract, including firms outside the embedding sample. Firm market equity is aggregated at `permco`; both fund and market weights sum to one.

For normalized portfolios, Active Share can be calculated efficiently as $1-\sum_a\min(w_{iat},w^{mkt}_{at})$. This is exactly equivalent to one-half of the absolute weight difference over the union of fund and market holdings.

In [ ]:
active_share_path = EXTENSION_ROOT / 'measures' / 's12_active_share_80q.parquet'
active_share_audit_path = EXTENSION_ROOT / 'manifests' / 'active_share_quarter_audit.parquet'

if active_share_path.exists() and active_share_audit_path.exists():
    active_share_panel = pd.read_parquet(active_share_path)
    active_share_audit = pd.read_parquet(active_share_audit_path)
    print('Loaded checkpointed Active Share panel.')
else:
    measure_rows = []
    audit_rows = []
    for year, quarter in QUARTERS:
        label = qlabel(year, quarter)
        holdings_path = REPLICATION_ROOT / 'processed' / label / 'combined_final_holdings.parquet'
        market_path = PROJECT_ROOT / 'data' / 'raw' / 'jkp_characteristics' / f'year={year}' / f'quarter={quarter}' / f'global_factor_paper_exact_{label}.parquet'

        holdings = pd.read_parquet(holdings_path)
        holdings = holdings.loc[holdings['source'].eq('s12'), ['investor_id','investor_key','investor_name','permco','dollar_holding']].copy()
        holdings['dollar_holding'] = pd.to_numeric(holdings['dollar_holding'], errors='coerce')
        holdings = holdings[holdings['dollar_holding'].gt(0)].copy()
        holdings['fund_total'] = holdings.groupby('investor_key')['dollar_holding'].transform('sum')
        holdings['fund_weight'] = holdings['dollar_holding'] / holdings['fund_total']

        market_raw = pd.read_parquet(market_path, columns=['permco','common','crsp_shrcd','me','me_company'])
        for column in ['permco','common','crsp_shrcd','me','me_company']:
            market_raw[column] = pd.to_numeric(market_raw[column], errors='coerce')
        market_raw = market_raw[(market_raw['common'].eq(1) | market_raw['crsp_shrcd'].isin([10,11])) & market_raw['permco'].notna()].copy()
        market_raw['firm_me'] = market_raw['me_company'].where(market_raw['me_company'].gt(0), market_raw['me'])
        market = (market_raw.dropna(subset=['firm_me']).loc[lambda x: x['firm_me'].gt(0)]
                  .groupby('permco', as_index=False)['firm_me'].max())
        market['permco'] = market['permco'].astype('int64')
        market['market_weight'] = market['firm_me'] / market['firm_me'].sum()

        joined = holdings.merge(market[['permco','market_weight']], on='permco', how='left', validate='many_to_one')
        joined['market_weight'] = joined['market_weight'].fillna(0.0)
        joined['weight_overlap'] = np.minimum(joined['fund_weight'], joined['market_weight'])
        fund_metrics = (joined.groupby(['investor_id','investor_key','investor_name'], as_index=False, dropna=False)
                        .agg(active_share=('weight_overlap', lambda x: 1.0 - x.sum()),
                             firms=('permco','nunique'), portfolio_value=('dollar_holding','sum'),
                             fund_weight_sum=('fund_weight','sum'),
                             market_weight_covered=('market_weight','sum')))
        fund_metrics.insert(0, 'quarter', quarter)
        fund_metrics.insert(0, 'year', year)
        fund_metrics['active_share_quintile'] = pd.qcut(
            fund_metrics['active_share'].rank(method='first'), 5, labels=[1,2,3,4,5]
        ).astype('int8')
        measure_rows.append(fund_metrics)
        audit_rows.append({
            'year': year, 'quarter': quarter, 'funds': len(fund_metrics),
            'fund_positions': len(holdings), 'market_firms': len(market),
            'mean_active_share': fund_metrics['active_share'].mean(),
            'median_active_share': fund_metrics['active_share'].median(),
            'minimum_active_share': fund_metrics['active_share'].min(),
            'maximum_active_share': fund_metrics['active_share'].max(),
            'maximum_fund_weight_error': (fund_metrics['fund_weight_sum'] - 1).abs().max(),
        })
        print(f'{label}: {len(fund_metrics):,} S12 funds, median Active Share {fund_metrics.active_share.median():.3f}')

    active_share_panel = pd.concat(measure_rows, ignore_index=True)
    active_share_audit = pd.DataFrame(audit_rows)
    save_parquet(active_share_panel, active_share_path)
    save_parquet(active_share_audit, active_share_audit_path)

assert active_share_panel['active_share'].between(0, 1 + 1e-10).all()
assert active_share_audit['maximum_fund_weight_error'].max() < 1e-8
display(active_share_panel['active_share'].describe(percentiles=[.01,.1,.25,.5,.75,.9,.99]).to_frame())
display(active_share_audit)
print(f'Saved Active Share panel: {active_share_path}')

## 12. Reported-index fallback without MFLINKS

The WRDS account can see MFLINKS metadata but cannot query its data. We therefore use only exact, dated S12 fund-ticker matches to CRSP through 2018. Once a `fundno`–`crsp_fundno` identity has been established, it may be carried forward for that same stable S12 identifier; no backward fill and no fuzzy name match are allowed. Funds without a prior exact match remain unknown. This reported label is secondary because coverage will decline for funds entering after 2018.

In [ ]:
reported_crosswalk_path = EXTENSION_ROOT / 'labels' / 's12_crsp_reported_index_80q.parquet'
reported_crosswalk_audit_path = EXTENSION_ROOT / 'manifests' / 'reported_index_linkage_audit.parquet'
source_root = EXTENSION_ROOT / 'labels' / 'linkage_sources'
source_root.mkdir(parents=True, exist_ok=True)

if not (reported_crosswalk_path.exists() and reported_crosswalk_audit_path.exists()):
    header_parts = []
    for start_year in range(2005, 2025, 5):
        end_year = min(start_year + 4, 2024)
        path = source_root / f'fund_hdr_hist_{start_year}_{end_year}.parquet'
        if path.exists():
            part = pd.read_parquet(path)
        else:
            part = db.raw_sql(f"""
                select crsp_fundno, chgdt, chgenddt, fund_name, ticker,
                       index_fund_flag, et_flag
                from crsp_q_mutualfunds.fund_hdr_hist
                where chgdt <= '{end_year}-12-31'
                  and chgenddt >= '{start_year}-01-01'
            """, date_cols=['chgdt','chgenddt'])
            save_parquet(part, path)
            print(f'Downloaded CRSP headers {start_year}-{end_year}: {len(part):,} rows')
        header_parts.append(part)
    headers = pd.concat(header_parts, ignore_index=True).drop_duplicates()
    headers['crsp_fundno'] = pd.to_numeric(headers['crsp_fundno'], errors='coerce')
    headers['chgdt'] = pd.to_datetime(headers['chgdt'])
    headers['chgenddt'] = pd.to_datetime(headers['chgenddt'])
    headers['ticker_clean'] = headers['ticker'].astype('string').str.upper().str.strip()

    exact_parts = []
    for fdate, ticker_frame in s12_fund_tickers.groupby('fdate', sort=True):
        current_headers = headers[headers['chgdt'].le(fdate) & headers['chgenddt'].ge(fdate)].copy()
        ticker_counts = current_headers.groupby('ticker_clean')['crsp_fundno'].nunique()
        unique_tickers = ticker_counts[ticker_counts.eq(1)].index
        current_headers = current_headers[current_headers['ticker_clean'].isin(unique_tickers)]
        current_headers = current_headers.drop_duplicates('ticker_clean')
        exact = ticker_frame.merge(
            current_headers[['ticker_clean','crsp_fundno']],
            left_on='fticker_clean', right_on='ticker_clean', how='inner', validate='many_to_one'
        )[['fdate','fundno','crsp_fundno']]
        exact_parts.append(exact)
    exact_map = pd.concat(exact_parts, ignore_index=True).drop_duplicates(['fdate','fundno'])
    save_parquet(exact_map, EXTENSION_ROOT / 'labels' / 's12_crsp_exact_ticker_matches.parquet')

    base = active_share_panel[['year','quarter','investor_id']].drop_duplicates().rename(columns={'investor_id':'fundno'})
    base['fdate'] = pd.PeriodIndex(year=base['year'], quarter=base['quarter'], freq='Q').to_timestamp(how='end').normalize()
    base['fundno'] = base['fundno'].astype('int64')
    exact_map['fundno'] = exact_map['fundno'].astype('int64')
    mapped = pd.merge_asof(
        base.sort_values(['fdate','fundno']),
        exact_map.rename(columns={'fdate':'identity_match_date'}).sort_values(['identity_match_date','fundno']),
        left_on='fdate', right_on='identity_match_date', by='fundno', direction='backward'
    )
    mapped['linkage_method'] = np.where(
        mapped['identity_match_date'].eq(mapped['fdate']), 'exact_dated_ticker',
        np.where(mapped['identity_match_date'].notna(), 'latest_prior_exact_ticker', 'unmatched')
    )

    labeled_parts = []
    for fdate, quarter_funds in mapped.groupby('fdate', sort=True):
        candidates = quarter_funds.merge(headers, on='crsp_fundno', how='left', validate='many_to_many')
        valid = candidates['chgdt'].isna() | (candidates['chgdt'].le(fdate) & candidates['chgenddt'].ge(fdate))
        labeled_parts.append(candidates.loc[valid])
    reported_index_panel = pd.concat(labeled_parts, ignore_index=True)
    reported_index_panel['flag_clean'] = reported_index_panel['index_fund_flag'].astype('string').str.upper().str.strip()
    reported_index_panel['header_matched'] = reported_index_panel['chgdt'].notna()
    reported_index_panel['reported_index'] = pd.array(
        np.where(reported_index_panel['header_matched'], reported_index_panel['flag_clean'].isin(['B','D','E']), pd.NA),
        dtype='boolean'
    )
    reported_index_panel['reported_index_codes'] = reported_index_panel['flag_clean'].where(
        reported_index_panel['flag_clean'].isin(['B','D','E']), ''
    )
    reported_index_panel['crsp_share_classes'] = reported_index_panel['crsp_fundno'].notna().astype('int8')
    reported_index_panel['conflicting_index_codes'] = False
    reported_index_panel = reported_index_panel.drop_duplicates(['year','quarter','fundno'])
    save_parquet(reported_index_panel, reported_crosswalk_path)

    reported_index_audit = (reported_index_panel.groupby(['year','quarter'], as_index=False)
        .agg(s12_funds=('fundno','size'), mflinks_matches=('crsp_fundno','count'),
             classified_funds=('reported_index','count'),
             reported_index_funds=('reported_index', lambda x: int(x.fillna(False).sum())),
             exact_ticker_links=('linkage_method', lambda x: int((x == 'exact_dated_ticker').sum())),
             carried_forward_links=('linkage_method', lambda x: int((x == 'latest_prior_exact_ticker').sum()))))
    reported_index_audit['mflinks_coverage'] = reported_index_audit['mflinks_matches'] / reported_index_audit['s12_funds']
    reported_index_audit['classification_coverage'] = reported_index_audit['classified_funds'] / reported_index_audit['s12_funds']
    save_parquet(reported_index_audit, reported_crosswalk_audit_path)
else:
    reported_index_panel = pd.read_parquet(reported_crosswalk_path)
    reported_index_audit = pd.read_parquet(reported_crosswalk_audit_path)

display(reported_index_audit)
print('Mean classification coverage:', f"{reported_index_audit.classification_coverage.mean():.1%}")

## 12A. Disabled MFLINKS alternative (subscription unavailable)

MFLINKS provides the defensible bridge: `mflink2` maps Thomson S12 `fundno` to the portfolio identifier `WFICN`, while `mflink1` maps `WFICN` to CRSP share classes. Each share class is then joined to the CRSP header record whose `[chgdt, chgenddt]` interval contains the S12 quarter date.

CRSP defines `B` as index-based, `D` as pure index, and `E` as enhanced index. All three indicate an index-oriented product, although we retain the exact code. A matched historical header with a null flag is classified as non-index; an unmatched fund remains unknown. Conflicting non-null codes across share classes are reported, never silently discarded.

In [ ]:
reported_crosswalk_path = EXTENSION_ROOT / 'labels' / 's12_crsp_reported_index_80q.parquet'
reported_crosswalk_audit_path = EXTENSION_ROOT / 'manifests' / 'reported_index_linkage_audit.parquet'

if True:
    if reported_crosswalk_path.exists() and reported_crosswalk_audit_path.exists():
        reported_index_panel = pd.read_parquet(reported_crosswalk_path)
        reported_index_audit = pd.read_parquet(reported_crosswalk_audit_path)
        print('Loaded the ticker-fallback reported-index crosswalk.')
    else:
        reported_index_panel = pd.DataFrame({'reported_index': pd.Series(dtype='boolean')})
        reported_index_audit = pd.DataFrame({'classification_coverage': pd.Series(dtype='float64')})
        print('MFLINKS query disabled. Run the preceding Reported-index fallback cell first.')
elif False:
    # Download restartable source pieces and perform the expensive interval join locally.
    try:
        db.raw_sql('select 1 as connection_test')
    except Exception:
        try:
            db.connection.close()
        except Exception:
            pass
        db = wrds.Connection()
        print('WRDS connection restored after the earlier timeout.')

    source_root = EXTENSION_ROOT / 'labels' / 'linkage_sources'
    source_root.mkdir(parents=True, exist_ok=True)
    mflinks_library = None
    for candidate_library in ['mfl', 'mflinks_all', 'mflinks_all_old']:
        try:
            db.raw_sql(f'select crsp_fundno, wficn from {candidate_library}.mflink1 limit 1')
            mflinks_library = candidate_library
            print(f'Using accessible MFLINKS library: {mflinks_library}')
            break
        except Exception as error:
            print(f'{candidate_library} is not queryable: {type(error).__name__}')
            try:
                db.connection.rollback()
            except Exception:
                pass
    if mflinks_library is None:
        raise PermissionError(
            'No MFLINKS schema is queryable with this WRDS account. '
            'The reported CRSP label cannot be extended defensibly beyond the available S12 ticker bridge.'
        )

    mflink1_path = source_root / 'mflink1.parquet'
    if mflink1_path.exists():
        mflink1 = pd.read_parquet(mflink1_path)
    else:
        mflink1 = db.raw_sql(f'select crsp_fundno, wficn from {mflinks_library}.mflink1')
        save_parquet(mflink1, mflink1_path)
        print(f'Downloaded MFLINK1: {len(mflink1):,} rows')

    mflink2_parts = []
    for year_part in range(2005, 2025):
        path = source_root / f'mflink2_{year_part}.parquet'
        if path.exists():
            part = pd.read_parquet(path)
        else:
            part = db.raw_sql(f"""
                select wficn, fdate, fundno, fundname, rdate, assets, ioc
                from {mflinks_library}.mflink2
                where fdate between '{year_part}-01-01' and '{year_part}-12-31'
            """, date_cols=['fdate','rdate'])
            save_parquet(part, path)
            print(f'Downloaded MFLINK2 {year_part}: {len(part):,} rows')
        mflink2_parts.append(part)
    mflink2 = pd.concat(mflink2_parts, ignore_index=True).drop_duplicates()

    header_parts = []
    for start_year in range(2005, 2025, 5):
        end_year = min(start_year + 4, 2024)
        path = source_root / f'fund_hdr_hist_{start_year}_{end_year}.parquet'
        if path.exists():
            part = pd.read_parquet(path)
        else:
            part = db.raw_sql(f"""
                select crsp_fundno, chgdt, chgenddt, fund_name,
                       ticker, index_fund_flag, et_flag
                from crsp_q_mutualfunds.fund_hdr_hist
                where chgdt <= '{end_year}-12-31'
                  and chgenddt >= '{start_year}-01-01'
            """, date_cols=['chgdt','chgenddt'])
            save_parquet(part, path)
            print(f'Downloaded CRSP headers {start_year}-{end_year}: {len(part):,} rows')
        header_parts.append(part)
    headers = pd.concat(header_parts, ignore_index=True).drop_duplicates()

    for frame, columns in [(mflink1, ['crsp_fundno','wficn']),
                           (mflink2, ['wficn','fundno']),
                           (headers, ['crsp_fundno'])]:
        for column in columns:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    mflink2['fdate'] = pd.to_datetime(mflink2['fdate'])
    headers['chgdt'] = pd.to_datetime(headers['chgdt'])
    headers['chgenddt'] = pd.to_datetime(headers['chgenddt'])

    linkage_parts = []
    for fdate, quarter_links in mflink2.groupby('fdate', sort=True):
        bridge = quarter_links.merge(mflink1, on='wficn', how='left', validate='many_to_many')
        candidates = bridge.merge(headers, on='crsp_fundno', how='left', validate='many_to_many')
        valid = candidates['chgdt'].isna() | (candidates['chgdt'].le(fdate) & candidates['chgenddt'].ge(fdate))
        matched = candidates.loc[valid].copy().rename(columns={'fund_name':'crsp_fund_name', 'ticker':'crsp_ticker'})
        linkage_parts.append(matched)
        print(f"{fdate:%Y-%m-%d}: {quarter_links['fundno'].nunique():,} S12 funds, {matched['crsp_fundno'].nunique():,} CRSP classes")
    linkage_detail = pd.concat(linkage_parts, ignore_index=True)
    save_parquet(linkage_detail, EXTENSION_ROOT / 'labels' / 's12_crsp_reported_index_detail.parquet')

    for column in ['fundno','wficn','crsp_fundno']:
        linkage_detail[column] = pd.to_numeric(linkage_detail[column], errors='coerce')
    linkage_detail['flag_clean'] = linkage_detail['index_fund_flag'].astype('string').str.upper().str.strip()
    linkage_detail['is_index_share_class'] = linkage_detail['flag_clean'].isin(['B','D','E'])
    linkage_detail['header_matched'] = linkage_detail['chgdt'].notna()

    def collapse_reported_label(group):
        codes = sorted(set(group.loc[group['flag_clean'].isin(['B','D','E']), 'flag_clean'].dropna()))
        header_matched = bool(group['header_matched'].any())
        return pd.Series({
            'wficns': group['wficn'].nunique(dropna=True),
            'crsp_share_classes': group['crsp_fundno'].nunique(dropna=True),
            'historical_headers_matched': int(group['header_matched'].sum()),
            'reported_index': bool(len(codes)) if header_matched else pd.NA,
            'reported_index_codes': ','.join(codes),
            'conflicting_index_codes': len(codes) > 1,
        })

    reported_index_panel = (linkage_detail.groupby(['fdate','fundno'], as_index=False, dropna=False)
                            .apply(collapse_reported_label, include_groups=False)
                            .reset_index())
    if 'level_2' in reported_index_panel:
        reported_index_panel = reported_index_panel.drop(columns='level_2')
    reported_index_panel['year'] = reported_index_panel['fdate'].dt.year
    reported_index_panel['quarter'] = reported_index_panel['fdate'].dt.quarter
    reported_index_panel['fundno'] = reported_index_panel['fundno'].astype('Int64')
    save_parquet(reported_index_panel, reported_crosswalk_path)

    base = active_share_panel[['year','quarter','investor_id']].drop_duplicates()
    diagnostic = base.merge(
        reported_index_panel, left_on=['year','quarter','investor_id'],
        right_on=['year','quarter','fundno'], how='left', validate='one_to_one'
    )
    reported_index_audit = (diagnostic.groupby(['year','quarter'], as_index=False)
        .agg(s12_funds=('investor_id','size'),
             mflinks_matches=('fundno','count'),
             classified_funds=('reported_index','count'),
             reported_index_funds=('reported_index', lambda x: int(x.fillna(False).sum())),
             conflicting_code_funds=('conflicting_index_codes', lambda x: int(x.fillna(False).sum()))))
    reported_index_audit['mflinks_coverage'] = reported_index_audit['mflinks_matches'] / reported_index_audit['s12_funds']
    reported_index_audit['classification_coverage'] = reported_index_audit['classified_funds'] / reported_index_audit['s12_funds']
    save_parquet(reported_index_audit, reported_crosswalk_audit_path)

display(reported_index_audit)
display(reported_index_panel['reported_index'].value_counts(dropna=False).rename_axis('reported_index').reset_index(name='fund_quarters'))
print('Mean classification coverage:', f"{reported_index_audit.classification_coverage.mean():.1%}")
print('Reported-index crosswalk saved to:', reported_crosswalk_path)

## 13. Join reported status to Active Share and validate economic ordering

This is not a definition-by-outcome test: the CRSP label is external and Active Share is holdings-based. A valid construction should nevertheless show lower Active Share among reported index funds on average. We report overlap and distributions without changing any classification threshold.

In [ ]:
fund_activeness_panel = active_share_panel.merge(
    reported_index_panel[['year','quarter','fundno','reported_index','reported_index_codes',
                          'crsp_share_classes','conflicting_index_codes']],
    left_on=['year','quarter','investor_id'],
    right_on=['year','quarter','fundno'], how='left', validate='one_to_one'
)
fund_activeness_panel['reported_style'] = pd.Series(
    'unmatched', index=fund_activeness_panel.index, dtype='string'
)
fund_activeness_panel.loc[fund_activeness_panel['reported_index'].eq(True).fillna(False), 'reported_style'] = 'reported_index'
fund_activeness_panel.loc[fund_activeness_panel['reported_index'].eq(False).fillna(False), 'reported_style'] = 'reported_non_index'
save_parquet(fund_activeness_panel, EXTENSION_ROOT / 'measures' / 's12_activeness_panel_partial.parquet')

reported_validation = (fund_activeness_panel.groupby('reported_style', as_index=False)
    .agg(fund_quarters=('investor_key','size'), funds=('investor_id','nunique'),
         mean_active_share=('active_share','mean'), median_active_share=('active_share','median'),
         low_active_share_quintile=('active_share_quintile', lambda x: float((x == 1).mean())),
         high_active_share_quintile=('active_share_quintile', lambda x: float((x == 5).mean()))))
display(reported_validation)
display(pd.crosstab(fund_activeness_panel['reported_style'], fund_activeness_panel['active_share_quintile'], normalize='index'))

## 14. Download and aggregate monthly firm returns

Return-adjusted turnover compares the current portfolio with the preceding portfolio carried forward by realized returns. We download only the required monthly US common-equity fields from the same JKP/CRSP source used in the replication, checkpoint one year at a time, select the primary security for each `permco`, and compound monthly total returns within each quarter. Partial-quarter observations are retained, while fund-level return coverage is measured explicitly in the next section.

In [ ]:
return_column_catalog = db.raw_sql("""
    select ordinal_position, column_name, data_type
    from information_schema.columns
    where table_schema = 'contrib_global_factor'
      and table_name = 'global_factor'
      and (position('ret' in lower(column_name)) > 0)
    order by ordinal_position
""")
display(return_column_catalog)
available_return_columns = set(return_column_catalog['column_name'].str.lower())
RETURN_COLUMN = next(
    (candidate for candidate in ['ret', 'eom_ret'] if candidate in available_return_columns),
    None,
)
if RETURN_COLUMN is None:
    raise KeyError('No total monthly-return field was found; inspect return_column_catalog before continuing.')
print(f'Using monthly total-return field: {RETURN_COLUMN}')
save_parquet(return_column_catalog, EXTENSION_ROOT / 'catalog' / 'global_factor_return_columns.parquet')

returns_root = EXTENSION_ROOT / 'measures' / 'firm_returns_v2_total_return'
returns_root.mkdir(parents=True, exist_ok=True)
annual_return_parts = []
for year_part in range(2005, 2025):
    path = returns_root / f'monthly_firm_returns_{year_part}.parquet'
    if path.exists():
        firm_month = pd.read_parquet(path)
    else:
        raw_returns = db.raw_sql(f"""
            select permno, permco, eom, {RETURN_COLUMN} as eom_ret, common, primary_sec, me
            from contrib_global_factor.global_factor
            where excntry = 'USA'
              and eom between '{year_part}-01-01' and '{year_part}-12-31'
        """, date_cols=['eom'])
        for column in ['permno','permco','eom_ret','common','primary_sec','me']:
            raw_returns[column] = pd.to_numeric(raw_returns[column], errors='coerce')
        raw_returns = raw_returns[raw_returns['permco'].notna() & raw_returns['common'].eq(1)].copy()
        raw_returns = raw_returns[raw_returns['eom_ret'].gt(-1) & raw_returns['eom_ret'].notna()].copy()
        raw_returns = raw_returns.sort_values(
            ['eom','permco','primary_sec','me'], ascending=[True,True,False,False]
        )
        firm_month = raw_returns.drop_duplicates(['eom','permco'])[['eom','permco','permno','eom_ret']].copy()
        firm_month['permco'] = firm_month['permco'].astype('int64')
        save_parquet(firm_month, path)
        print(f'Downloaded {year_part}: {len(firm_month):,} firm-month returns')
    annual_return_parts.append(firm_month)

monthly_firm_returns = pd.concat(annual_return_parts, ignore_index=True)
monthly_firm_returns['eom'] = pd.to_datetime(monthly_firm_returns['eom'])
monthly_firm_returns['year'] = monthly_firm_returns['eom'].dt.year
monthly_firm_returns['quarter'] = monthly_firm_returns['eom'].dt.quarter
quarterly_firm_returns = (monthly_firm_returns.groupby(['year','quarter','permco'], as_index=False)
    .agg(quarter_return=('eom_ret', lambda x: float(np.prod(1.0 + x) - 1.0)),
         return_months=('eom_ret','count')))
save_parquet(quarterly_firm_returns, returns_root / 'quarterly_firm_returns_2005_2024.parquet')
display(quarterly_firm_returns[['quarter_return','return_months']].describe(percentiles=[.01,.5,.99]))
print(f'Firm-quarter returns: {len(quarterly_firm_returns):,}')

## 15. Construct return-adjusted turnover and its margins

For every fund observed in consecutive quarters, prior weights are grown by the intervening firm returns and renormalized. Total turnover is one-half the absolute difference between current and buy-and-hold weights. The extensive component contains entries and exits; the intensive component contains reweighting among continuing positions. These components sum exactly to total turnover.

A fund-quarter is primary-sample valid when returns cover at least 90% of the preceding portfolio weight. Missing returns are assigned zero only for constructing the diagnostic value; observations below the coverage threshold are flagged and excluded from quintile formation.

In [ ]:
turnover_path = EXTENSION_ROOT / 'measures' / 's12_return_adjusted_turnover_80q_v2_total_return.parquet'
turnover_parts_root = EXTENSION_ROOT / 'measures' / 'turnover_quarter_parts_v2_total_return'
turnover_parts_root.mkdir(parents=True, exist_ok=True)

turnover_parts = []
for position in range(1, len(QUARTERS)):
    prior_year, prior_quarter = QUARTERS[position - 1]
    year, quarter = QUARTERS[position]
    label = qlabel(year, quarter)
    part_path = turnover_parts_root / f'turnover_{label}.parquet'
    if part_path.exists():
        metrics = pd.read_parquet(part_path)
        turnover_parts.append(metrics)
        continue

    def s12_weights(y, q):
        frame = pd.read_parquet(REPLICATION_ROOT / 'processed' / qlabel(y, q) / 'combined_final_holdings.parquet')
        frame = frame.loc[frame['source'].eq('s12'), ['investor_id','investor_key','investor_name','permco','dollar_holding']].copy()
        frame['fund_total'] = frame.groupby('investor_key')['dollar_holding'].transform('sum')
        frame['weight'] = frame['dollar_holding'] / frame['fund_total']
        return frame[['investor_id','investor_key','investor_name','permco','weight']]

    prior = s12_weights(prior_year, prior_quarter)
    current = s12_weights(year, quarter)
    common_funds = set(prior['investor_key']).intersection(current['investor_key'])
    prior = prior[prior['investor_key'].isin(common_funds)].copy()
    current = current[current['investor_key'].isin(common_funds)].copy()
    returns = quarterly_firm_returns.loc[
        quarterly_firm_returns['year'].eq(year) & quarterly_firm_returns['quarter'].eq(quarter),
        ['permco','quarter_return','return_months']
    ]
    prior = prior.merge(returns, on='permco', how='left', validate='many_to_one')
    prior['covered_weight_piece'] = prior['weight'] * prior['quarter_return'].notna()
    prior['gross_return'] = 1.0 + prior['quarter_return'].fillna(0.0)
    prior['bh_value'] = prior['weight'] * prior['gross_return'].clip(lower=0.0)
    prior['bh_total'] = prior.groupby('investor_key')['bh_value'].transform('sum')
    prior['bh_weight'] = prior['bh_value'] / prior['bh_total']

    positions = prior[['investor_id','investor_key','investor_name','permco','bh_weight','covered_weight_piece']].merge(
        current[['investor_key','permco','weight']].rename(columns={'weight':'current_weight'}),
        on=['investor_key','permco'], how='outer', validate='one_to_one'
    )
    positions['bh_weight'] = positions['bh_weight'].fillna(0.0)
    positions['current_weight'] = positions['current_weight'].fillna(0.0)
    positions['covered_weight_piece'] = positions['covered_weight_piece'].fillna(0.0)
    positions['absolute_change'] = (positions['current_weight'] - positions['bh_weight']).abs()
    positions['entry_piece'] = np.where(positions['bh_weight'].eq(0), positions['current_weight'], 0.0)
    positions['exit_piece'] = np.where(positions['current_weight'].eq(0), positions['bh_weight'], 0.0)
    positions['intensive_piece'] = np.where(
        positions['bh_weight'].gt(0) & positions['current_weight'].gt(0), positions['absolute_change'], 0.0
    )
    metrics = (positions.groupby('investor_key', as_index=False)
        .agg(investor_id=('investor_id','first'), investor_name=('investor_name','first'),
             return_coverage=('covered_weight_piece','sum'),
             return_adjusted_turnover=('absolute_change', lambda x: 0.5 * x.sum()),
             extensive_turnover=('entry_piece', lambda x: 0.5 * x.sum()),
             exit_turnover=('exit_piece', lambda x: 0.5 * x.sum()),
             intensive_turnover=('intensive_piece', lambda x: 0.5 * x.sum()),
             positions_union=('permco','nunique')))
    metrics['entry_turnover'] = metrics['extensive_turnover']
    metrics['extensive_turnover'] = metrics['entry_turnover'] + metrics['exit_turnover']
    metrics['turnover_valid'] = metrics['return_coverage'].ge(0.90)
    metrics['decomposition_error'] = (
        metrics['return_adjusted_turnover'] - metrics['extensive_turnover'] - metrics['intensive_turnover']
    ).abs()
    metrics.insert(0, 'quarter', quarter)
    metrics.insert(0, 'year', year)
    valid = metrics['turnover_valid']
    metrics['turnover_quintile'] = pd.Series(pd.NA, index=metrics.index, dtype='Int8')
    if valid.sum() >= 5:
        metrics.loc[valid, 'turnover_quintile'] = pd.qcut(
            metrics.loc[valid, 'return_adjusted_turnover'].rank(method='first'), 5, labels=[1,2,3,4,5]
        ).astype('int8')
    assert metrics['decomposition_error'].max() < 1e-8
    save_parquet(metrics, part_path)
    turnover_parts.append(metrics)
    print(f'{label}: {len(metrics):,} continuing funds; {valid.mean():.1%} pass return coverage')

turnover_panel = pd.concat(turnover_parts, ignore_index=True)
save_parquet(turnover_panel, turnover_path)
display(turnover_panel.groupby('turnover_valid').size().rename('fund_quarters').reset_index())
display(turnover_panel.loc[turnover_panel['turnover_valid'], ['return_adjusted_turnover','extensive_turnover','intensive_turnover','return_coverage']].describe(percentiles=[.01,.1,.5,.9,.99]))
print('Maximum decomposition error:', turnover_panel['decomposition_error'].max())
print('Saved turnover panel:', turnover_path)

## 16. Combine the activeness measures and define behavioral groups

The primary extension does not force reported labels onto unmatched funds. Instead, each valid fund-quarter is split at that quarter's median Active Share and median return-adjusted turnover. This creates four economically distinct, reasonably sized samples:

- **Passive-like:** low Active Share and low turnover.
- **Stable stock picker:** high Active Share and low turnover.
- **Benchmark rebalancer:** low Active Share and high turnover.
- **Tactical active:** high Active Share and high turnover.

Median groups are used for embedding estimation to avoid a mechanical sample-size disadvantage. First and fifth quintiles are retained as stricter tail definitions for robustness. Reported CRSP status remains an external validation field.

In [ ]:
turnover_panel = pd.read_parquet(
    EXTENSION_ROOT / 'measures' / 's12_return_adjusted_turnover_80q_v2_total_return.parquet'
)
master_activeness = fund_activeness_panel.merge(
    turnover_panel[['year','quarter','investor_key','return_coverage','return_adjusted_turnover',
                    'extensive_turnover','entry_turnover','exit_turnover','intensive_turnover',
                    'turnover_valid','turnover_quintile']],
    on=['year','quarter','investor_key'], how='left', validate='one_to_one'
)
master_activeness['behavioral_group'] = pd.Series('turnover_unavailable', index=master_activeness.index, dtype='string')
valid = master_activeness['turnover_valid'].fillna(False)
quarter_medians = (master_activeness.loc[valid].groupby(['year','quarter'], as_index=False)
    .agg(active_share_median=('active_share','median'), turnover_median=('return_adjusted_turnover','median')))
master_activeness = master_activeness.merge(quarter_medians, on=['year','quarter'], how='left', validate='many_to_one')
low_as = master_activeness['active_share'].le(master_activeness['active_share_median'])
low_to = master_activeness['return_adjusted_turnover'].le(master_activeness['turnover_median'])
master_activeness.loc[valid & low_as & low_to, 'behavioral_group'] = 'passive_like'
master_activeness.loc[valid & ~low_as & low_to, 'behavioral_group'] = 'stable_stock_picker'
master_activeness.loc[valid & low_as & ~low_to, 'behavioral_group'] = 'benchmark_rebalancer'
master_activeness.loc[valid & ~low_as & ~low_to, 'behavioral_group'] = 'tactical_active'
master_activeness['strict_tail_group'] = pd.Series('middle', index=master_activeness.index, dtype='string')
master_activeness.loc[valid & master_activeness['active_share_quintile'].eq(1) & master_activeness['turnover_quintile'].eq(1), 'strict_tail_group'] = 'passive_like_tail'
master_activeness.loc[valid & master_activeness['active_share_quintile'].eq(5) & master_activeness['turnover_quintile'].eq(1), 'strict_tail_group'] = 'stable_stock_picker_tail'
master_activeness.loc[valid & master_activeness['active_share_quintile'].eq(1) & master_activeness['turnover_quintile'].eq(5), 'strict_tail_group'] = 'benchmark_rebalancer_tail'
master_activeness.loc[valid & master_activeness['active_share_quintile'].eq(5) & master_activeness['turnover_quintile'].eq(5), 'strict_tail_group'] = 'tactical_active_tail'
save_parquet(master_activeness, EXTENSION_ROOT / 'measures' / 's12_activeness_master_80q.parquet')

group_summary = (master_activeness.groupby('behavioral_group', as_index=False)
    .agg(fund_quarters=('investor_key','size'), funds=('investor_id','nunique'),
         mean_active_share=('active_share','mean'), mean_turnover=('return_adjusted_turnover','mean'),
         mean_extensive=('extensive_turnover','mean'), mean_intensive=('intensive_turnover','mean'),
         reported_index_rate=('reported_index', lambda x: float(x.dropna().mean()) if x.notna().any() else np.nan),
         reported_label_coverage=('reported_index', lambda x: float(x.notna().mean()))))
display(group_summary)
print('Active Share–turnover correlation:', master_activeness.loc[valid, ['active_share','return_adjusted_turnover']].corr().iloc[0,1])
display(master_activeness['strict_tail_group'].value_counts().rename_axis('strict_tail_group').reset_index(name='fund_quarters'))

## 17. Audit embedding-sample feasibility

Before fitting models, this cell joins the classifications back to the immutable S12 holdings and reports funds, firms, positions, and network degrees for every quarter and behavioral group. A group must have sufficient investor and firm support after the same iterative degree restriction used in the replication; otherwise its embedding result would mostly reflect a thin-network artifact.

In [ ]:
feasibility_rows = []
embedding_groups = ['passive_like','stable_stock_picker','benchmark_rebalancer','tactical_active']
for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    holdings = pd.read_parquet(REPLICATION_ROOT / 'processed' / label / 'combined_final_holdings.parquet')
    holdings = holdings[holdings['source'].eq('s12')].copy()
    labels = master_activeness.loc[
        master_activeness['year'].eq(year) & master_activeness['quarter'].eq(quarter),
        ['investor_key','behavioral_group']
    ]
    holdings = holdings.merge(labels, on='investor_key', how='inner', validate='many_to_one')
    for group in embedding_groups:
        subset = holdings[holdings['behavioral_group'].eq(group)]
        investor_degree = subset.groupby('investor_key')['permco'].nunique()
        firm_degree = subset.groupby('permco')['investor_key'].nunique()
        feasibility_rows.append({
            'year': year, 'quarter': quarter, 'behavioral_group': group,
            'positions': len(subset), 'funds': subset['investor_key'].nunique(),
            'firms': subset['permco'].nunique(),
            'median_firms_per_fund': investor_degree.median() if len(investor_degree) else np.nan,
            'firms_with_at_least_20_funds': int(firm_degree.ge(20).sum()),
            'funds_with_at_least_20_firms': int(investor_degree.ge(20).sum()),
        })
feasibility = pd.DataFrame(feasibility_rows)
save_parquet(feasibility, EXTENSION_ROOT / 'manifests' / 'behavioral_group_feasibility.parquet')
display(feasibility.groupby('behavioral_group', as_index=False).agg(
    quarters=('quarter','size'), minimum_funds=('funds','min'), median_funds=('funds','median'),
    minimum_firms=('firms','min'), median_firms=('firms','median'),
    minimum_dense_firms=('firms_with_at_least_20_funds','min'),
    median_dense_firms=('firms_with_at_least_20_funds','median'),
))
display(feasibility.sort_values('funds').head(20))

## 18. Build comparable behavioral-group matrices

For each quarter, the estimable asset universe is the intersection of firms held by at least five funds in **every** behavioral group. Within that common universe, funds must retain at least ten positions. We then remove any firm absent from even one post-filter group. The resulting firm mapping is identical across all four models in a quarter, preventing performance differences from being driven by different valuation samples. These thresholds are extension-specific responses to splitting the already-filtered parent network; the original replication remains unchanged.

In [ ]:
from scipy import sparse

GROUP_MATRIX_ROOT = EXTENSION_ROOT / 'samples' / 'behavioral_group_matrices'
GROUP_MINIMUM_FIRM_DEGREE = 5
GROUP_MINIMUM_FUND_DEGREE = 10
group_matrix_audit_rows = []

for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    quarter_root = GROUP_MATRIX_ROOT / label
    audit_path = quarter_root / 'matrix_audit.parquet'
    if audit_path.exists():
        group_matrix_audit_rows.append(pd.read_parquet(audit_path))
        continue
    holdings = pd.read_parquet(REPLICATION_ROOT / 'processed' / label / 'combined_final_holdings.parquet')
    holdings = holdings[holdings['source'].eq('s12')].copy()
    labels = master_activeness.loc[
        master_activeness['year'].eq(year) & master_activeness['quarter'].eq(quarter),
        ['investor_key','behavioral_group']
    ]
    holdings = holdings.merge(labels, on='investor_key', how='inner', validate='many_to_one')
    eligible_sets = []
    for group in embedding_groups:
        degree = holdings.loc[holdings['behavioral_group'].eq(group)].groupby('permco')['investor_key'].nunique()
        eligible_sets.append(set(degree[degree.ge(GROUP_MINIMUM_FIRM_DEGREE)].index))
    common_firms = set.intersection(*eligible_sets)
    filtered_groups = {}
    represented_sets = []
    for group in embedding_groups:
        subset = holdings[holdings['behavioral_group'].eq(group) & holdings['permco'].isin(common_firms)].copy()
        fund_degree = subset.groupby('investor_key')['permco'].nunique()
        subset = subset[subset['investor_key'].isin(fund_degree[fund_degree.ge(GROUP_MINIMUM_FUND_DEGREE)].index)].copy()
        filtered_groups[group] = subset
        represented_sets.append(set(subset['permco']))
    final_common_firms = sorted(set.intersection(*represented_sets))
    firm_names = (holdings[holdings['permco'].isin(final_common_firms)][['permco','firm_name']]
                  .drop_duplicates('permco').sort_values('permco').reset_index(drop=True))
    firm_names['firm_index'] = np.arange(len(firm_names), dtype='int32')
    save_parquet(firm_names, quarter_root / 'firm_mapping.parquet')
    quarter_audit = []
    for group, subset in filtered_groups.items():
        subset = subset[subset['permco'].isin(final_common_firms)].copy()
        investors = subset[['investor_key','investor_id','investor_name']].drop_duplicates('investor_key').sort_values('investor_key').reset_index(drop=True)
        investors['investor_index'] = np.arange(len(investors), dtype='int32')
        indexed = (subset.merge(investors[['investor_key','investor_index']], on='investor_key', validate='many_to_one')
                   .merge(firm_names[['permco','firm_index']], on='permco', validate='many_to_one'))
        shape = (len(investors), len(firm_names))
        rows = indexed['investor_index'].to_numpy(dtype='int32')
        columns = indexed['firm_index'].to_numpy(dtype='int32')
        binary = sparse.csr_matrix((np.ones(len(indexed), dtype='float32'), (rows, columns)), shape=shape)
        log_values = np.log(indexed['dollar_holding'].to_numpy(dtype='float64'))
        log_matrix = sparse.csr_matrix((log_values.astype('float32'), (rows, columns)), shape=shape)
        group_root = quarter_root / group
        group_root.mkdir(parents=True, exist_ok=True)
        sparse.save_npz(group_root / 'binary.npz', binary, compressed=True)
        sparse.save_npz(group_root / 'log_holdings.npz', log_matrix, compressed=True)
        save_parquet(investors, group_root / 'investor_mapping.parquet')
        quarter_audit.append({'year':year,'quarter':quarter,'behavioral_group':group,
                              'investors':shape[0],'firms':shape[1],'positions':binary.nnz,
                              'density':binary.nnz / np.prod(shape),
                              'minimum_fund_degree':int(np.diff(binary.indptr).min()),
                              'minimum_firm_degree':int(np.diff(binary.tocsc().indptr).min())})
    quarter_audit = pd.DataFrame(quarter_audit)
    save_parquet(quarter_audit, audit_path)
    group_matrix_audit_rows.append(quarter_audit)
    print(f'{label}: {len(final_common_firms):,} common firms')

group_matrix_audit = pd.concat(group_matrix_audit_rows, ignore_index=True)
save_parquet(group_matrix_audit, GROUP_MATRIX_ROOT / 'eighty_quarter_matrix_audit.parquet')
display(group_matrix_audit.groupby('behavioral_group', as_index=False).agg(
    quarters=('quarter','size'), minimum_investors=('investors','min'), median_investors=('investors','median'),
    minimum_firms=('firms','min'), median_firms=('firms','median'),
    minimum_realized_firm_degree=('minimum_firm_degree','min'),
))

## 19. Fit RS-Binary and RS-L-Min behavioral embeddings

This reproduces the two primary PCA-based recommender-system constructions from the replication at 10 components; the saved embedding contains both the 4D and 10D specifications. RS-Binary uses portfolio incidence. RS-L-Min removes investor and firm fixed effects from observed log dollar holdings and fills unobserved positions with each investor's minimum residual before PCA. Every fit is checkpointed by quarter and group.

In [ ]:
from sklearn.decomposition import PCA
import time

GROUP_EMBEDDING_ROOT = EXTENSION_ROOT / 'benchmarks' / 'behavioral_embeddings'
embedding_fit_rows = []
for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    firm_mapping = pd.read_parquet(GROUP_MATRIX_ROOT / label / 'firm_mapping.parquet')
    for group in embedding_groups:
        group_matrix_root = GROUP_MATRIX_ROOT / label / group
        binary = sparse.load_npz(group_matrix_root / 'binary.npz')
        log_matrix = sparse.load_npz(group_matrix_root / 'log_holdings.npz')
        for model in ['rs_binary','rs_l_min']:
            output_root = GROUP_EMBEDDING_ROOT / label / group / model
            embedding_path = output_root / 'asset_embeddings_10d.parquet'
            fit_path = output_root / 'fit_summary.parquet'
            if embedding_path.exists() and fit_path.exists():
                embedding_fit_rows.append(pd.read_parquet(fit_path))
                continue
            if model == 'rs_binary':
                dense = binary.toarray().astype('float32')
            else:
                coo = log_matrix.tocoo()
                investor_counts = np.bincount(coo.row, minlength=log_matrix.shape[0])
                investor_fe = np.bincount(coo.row, weights=coo.data, minlength=log_matrix.shape[0]) / investor_counts
                investor_demeaned = coo.data - investor_fe[coo.row]
                firm_counts = np.bincount(coo.col, minlength=log_matrix.shape[1])
                firm_fe = np.bincount(coo.col, weights=investor_demeaned, minlength=log_matrix.shape[1]) / firm_counts
                residual = investor_demeaned - firm_fe[coo.col]
                investor_minimum = np.full(log_matrix.shape[0], np.inf)
                np.minimum.at(investor_minimum, coo.row, residual)
                dense = np.repeat(investor_minimum[:,None], log_matrix.shape[1], axis=1).astype('float32')
                dense[coo.row, coo.col] = residual.astype('float32')
            started = time.perf_counter()
            pca = PCA(n_components=10, svd_solver='randomized', random_state=202451).fit(dense)
            assets = (pca.components_.T * np.sqrt(dense.shape[1])).astype('float32')
            embeddings = firm_mapping[['permco','firm_name','firm_index']].copy()
            for component in range(10):
                embeddings[f'embedding_{component + 1:03d}'] = assets[:,component]
            fit = pd.DataFrame({'year':[year],'quarter':[quarter],'behavioral_group':[group],'model':[model],
                                'investors':[dense.shape[0]],'firms':[dense.shape[1]],
                                'positions':[binary.nnz],'fit_seconds':[time.perf_counter()-started],
                                'explained_variance_4d':[pca.explained_variance_ratio_[:4].sum()],
                                'explained_variance_10d':[pca.explained_variance_ratio_.sum()]})
            save_parquet(embeddings, embedding_path)
            save_parquet(fit, fit_path)
            embedding_fit_rows.append(fit)
            del dense
        print(f'{label} {group}: complete')
embedding_fit_inventory = pd.concat(embedding_fit_rows, ignore_index=True)
save_parquet(embedding_fit_inventory, GROUP_EMBEDDING_ROOT / 'fit_inventory.parquet')
display(embedding_fit_inventory.groupby(['behavioral_group','model'], as_index=False).agg(
    quarters=('quarter','size'), mean_investors=('investors','mean'), mean_firms=('firms','mean'),
    mean_fit_seconds=('fit_seconds','mean'), mean_explained_variance_10d=('explained_variance_10d','mean')))

## 20. Common-sample relative-valuation benchmark

Every behavioral embedding is evaluated on the same firms and the replication's saved train/test assignment. Relative valuation is rebuilt fresh within each quarter's common firm sample as the residual from log market equity on log book equity. Book equity is used only to construct the target and is excluded from every predictor set. Predictor scaling and ridge selection are fitted on training firms only.

The original combined S12–S34 RS-Binary and RS-L-Min embeddings are reevaluated on this identical restricted sample. They provide a fair benchmark rather than comparing behavioral models on fewer firms with the original headline results.

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

GROUP_BENCHMARK_ROOT = EXTENSION_ROOT / 'benchmarks' / 'behavioral_relative_valuation'
GROUP_BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
RIDGE_ALPHAS_EXTENSION = np.logspace(-6, 6, 49)
benchmark_parts = []

for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    result_path = GROUP_BENCHMARK_ROOT / label / 'results.parquet'
    if result_path.exists():
        benchmark_parts.append(pd.read_parquet(result_path))
        continue
    common_firms = pd.read_parquet(GROUP_MATRIX_ROOT / label / 'firm_mapping.parquet')[['permco']]
    panel = pd.read_parquet(REPLICATION_ROOT / 'benchmarks' / label / 'evaluation_sample_and_split.parquet')
    panel = panel.merge(common_firms, on='permco', how='inner', validate='one_to_one').copy()
    panel = panel[panel['jkp_firm_market_equity'].gt(0) & panel['jkp_book_equity'].gt(0)].copy()
    panel['log_me_extension'] = np.log(panel['jkp_firm_market_equity'])
    panel['log_be_extension'] = np.log(panel['jkp_book_equity'])
    target_model = LinearRegression().fit(panel[['log_be_extension']], panel['log_me_extension'])
    panel['relative_valuation_extension'] = panel['log_me_extension'] - target_model.predict(panel[['log_be_extension']])

    model_frames = {
        'combined_s12_s34__rs_binary': panel[['permco'] + [f'rs_binary_embedding_{i:03d}' for i in range(1,11)]].rename(
            columns={f'rs_binary_embedding_{i:03d}':f'embedding_{i:03d}' for i in range(1,11)}),
        'combined_s12_s34__rs_l_min': panel[['permco'] + [f'rs_l_min_embedding_{i:03d}' for i in range(1,11)]].rename(
            columns={f'rs_l_min_embedding_{i:03d}':f'embedding_{i:03d}' for i in range(1,11)}),
    }
    for group in embedding_groups:
        for model in ['rs_binary','rs_l_min']:
            model_frames[f'{group}__{model}'] = pd.read_parquet(
                GROUP_EMBEDDING_ROOT / label / group / model / 'asset_embeddings_10d.parquet'
            )

    train_mask = panel['sample'].eq('train').to_numpy()
    test_mask = panel['sample'].eq('test').to_numpy()
    y = panel['relative_valuation_extension'].to_numpy(dtype='float64')
    y_train, y_test = y[train_mask], y[test_mask]
    rows = []
    for model_name, embeddings in model_frames.items():
        joined = panel[['permco']].merge(embeddings, on='permco', how='left', validate='one_to_one')
        if joined.filter(like='embedding_').isna().any().any():
            raise ValueError(f'{label} {model_name} lacks a common-sample embedding.')
        for dimension in [4,10]:
            columns = [f'embedding_{i:03d}' for i in range(1, dimension + 1)]
            X = joined[columns].to_numpy(dtype='float64')
            scaler = StandardScaler().fit(X[train_mask])
            X_train, X_test = scaler.transform(X[train_mask]), scaler.transform(X[test_mask])
            cv = KFold(n_splits=10, shuffle=True, random_state=202451)
            ridge = RidgeCV(alphas=RIDGE_ALPHAS_EXTENSION, cv=cv, scoring='neg_mean_squared_error').fit(X_train, y_train)
            prediction = ridge.predict(X_test)
            residual = y_test - prediction
            group_name, embedding_model = model_name.split('__')
            rows.append({'year':year,'quarter':quarter,'investor_group':group_name,
                         'embedding_model':embedding_model,'dimension':dimension,
                         'firms':len(panel),'training_firms':int(train_mask.sum()),'test_firms':int(test_mask.sum()),
                         'selected_alpha':float(ridge.alpha_),
                         'test_variance_r2':float(1 - np.var(residual) / np.var(y_test)),
                         'test_conventional_r2':float(r2_score(y_test, prediction)),
                         'test_prediction_correlation':float(np.corrcoef(y_test, prediction)[0,1]),
                         'book_equity_slope':float(target_model.coef_[0])})
    results = pd.DataFrame(rows)
    save_parquet(results, result_path)
    benchmark_parts.append(results)
    print(f'{label}: {len(panel):,} common valuation firms, {len(results)} model results')

behavioral_benchmark = pd.concat(benchmark_parts, ignore_index=True)
behavioral_summary = (behavioral_benchmark.groupby(['investor_group','embedding_model','dimension'], as_index=False)
    .agg(quarters=('quarter','size'), mean_test_variance_r2=('test_variance_r2','mean'),
         median_test_variance_r2=('test_variance_r2','median'),
         standard_deviation_r2=('test_variance_r2','std'), mean_firms=('firms','mean'),
         mean_selected_alpha=('selected_alpha','mean')))
save_parquet(behavioral_benchmark, GROUP_BENCHMARK_ROOT / 'eighty_quarter_results.parquet')
save_parquet(behavioral_summary, GROUP_BENCHMARK_ROOT / 'summary.parquet')
display(behavioral_summary.sort_values(['embedding_model','dimension','mean_test_variance_r2'], ascending=[True,True,False]))

## 21. All-S12 control on the identical common firm universe

This is the extension's primary baseline. It pools all turnover-eligible S12 funds from the four behavioral groups, restricts them to the exact quarterly firm mapping used above, and applies the same ten-position fund requirement. RS-Binary and RS-L-Min are fitted and evaluated exactly as in Sections 19–20. The combined S12–S34 model remains a separate replication reference.

In [ ]:
ALL_S12_ROOT = EXTENSION_ROOT / 'benchmarks' / 'all_s12_control'
all_s12_result_parts = []
all_s12_fit_parts = []
for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    output_root = ALL_S12_ROOT / label
    result_path = output_root / 'valuation_results.parquet'
    fit_path = output_root / 'fit_summary.parquet'
    if result_path.exists() and fit_path.exists():
        all_s12_result_parts.append(pd.read_parquet(result_path))
        all_s12_fit_parts.append(pd.read_parquet(fit_path))
        continue

    firm_mapping = pd.read_parquet(GROUP_MATRIX_ROOT / label / 'firm_mapping.parquet')
    holdings = pd.read_parquet(REPLICATION_ROOT / 'processed' / label / 'combined_final_holdings.parquet')
    holdings = holdings[holdings['source'].eq('s12') & holdings['permco'].isin(firm_mapping['permco'])].copy()
    valid_funds = set(master_activeness.loc[
        master_activeness['year'].eq(year) & master_activeness['quarter'].eq(quarter)
        & master_activeness['behavioral_group'].isin(embedding_groups), 'investor_key'
    ])
    holdings = holdings[holdings['investor_key'].isin(valid_funds)].copy()
    degree = holdings.groupby('investor_key')['permco'].nunique()
    holdings = holdings[holdings['investor_key'].isin(degree[degree.ge(GROUP_MINIMUM_FUND_DEGREE)].index)].copy()
    investors = holdings[['investor_key','investor_id','investor_name']].drop_duplicates('investor_key').sort_values('investor_key').reset_index(drop=True)
    investors['investor_index'] = np.arange(len(investors), dtype='int32')
    indexed = (holdings.merge(investors[['investor_key','investor_index']], on='investor_key', validate='many_to_one')
               .merge(firm_mapping[['permco','firm_index']], on='permco', validate='many_to_one'))
    rows = indexed['investor_index'].to_numpy(dtype='int32')
    columns = indexed['firm_index'].to_numpy(dtype='int32')
    shape = (len(investors), len(firm_mapping))
    binary = sparse.csr_matrix((np.ones(len(indexed), dtype='float32'), (rows, columns)), shape=shape)
    log_values = np.log(indexed['dollar_holding'].to_numpy(dtype='float64'))
    log_matrix = sparse.csr_matrix((log_values.astype('float32'), (rows, columns)), shape=shape)

    embedding_frames = {}
    fit_rows = []
    for model in ['rs_binary','rs_l_min']:
        if model == 'rs_binary':
            dense = binary.toarray().astype('float32')
        else:
            coo = log_matrix.tocoo()
            investor_counts = np.bincount(coo.row, minlength=shape[0])
            investor_fe = np.bincount(coo.row, weights=coo.data, minlength=shape[0]) / investor_counts
            investor_demeaned = coo.data - investor_fe[coo.row]
            firm_counts = np.bincount(coo.col, minlength=shape[1])
            firm_fe = np.bincount(coo.col, weights=investor_demeaned, minlength=shape[1]) / firm_counts
            residual = investor_demeaned - firm_fe[coo.col]
            investor_minimum = np.full(shape[0], np.inf)
            np.minimum.at(investor_minimum, coo.row, residual)
            dense = np.repeat(investor_minimum[:,None], shape[1], axis=1).astype('float32')
            dense[coo.row, coo.col] = residual.astype('float32')
        started = time.perf_counter()
        pca = PCA(n_components=10, svd_solver='randomized', random_state=202451).fit(dense)
        assets = (pca.components_.T * np.sqrt(shape[1])).astype('float32')
        frame = firm_mapping[['permco','firm_name','firm_index']].copy()
        for component in range(10):
            frame[f'embedding_{component + 1:03d}'] = assets[:,component]
        embedding_frames[model] = frame
        save_parquet(frame, output_root / model / 'asset_embeddings_10d.parquet')
        fit_rows.append({'year':year,'quarter':quarter,'model':model,'investors':shape[0],
                         'firms':shape[1],'positions':binary.nnz,'fit_seconds':time.perf_counter()-started,
                         'explained_variance_4d':pca.explained_variance_ratio_[:4].sum(),
                         'explained_variance_10d':pca.explained_variance_ratio_.sum()})
        del dense

    panel = pd.read_parquet(REPLICATION_ROOT / 'benchmarks' / label / 'evaluation_sample_and_split.parquet')
    panel = panel.merge(firm_mapping[['permco']], on='permco', how='inner', validate='one_to_one').copy()
    panel['log_me_extension'] = np.log(panel['jkp_firm_market_equity'])
    panel['log_be_extension'] = np.log(panel['jkp_book_equity'])
    target_model = LinearRegression().fit(panel[['log_be_extension']], panel['log_me_extension'])
    y = (panel['log_me_extension'] - target_model.predict(panel[['log_be_extension']])).to_numpy()
    train_mask = panel['sample'].eq('train').to_numpy()
    test_mask = panel['sample'].eq('test').to_numpy()
    result_rows = []
    for model, frame in embedding_frames.items():
        X_all = panel[['permco']].merge(frame, on='permco', validate='one_to_one')
        for dimension in [4,10]:
            columns_x = [f'embedding_{i:03d}' for i in range(1,dimension + 1)]
            X = X_all[columns_x].to_numpy(dtype='float64')
            scaler = StandardScaler().fit(X[train_mask])
            X_train, X_test = scaler.transform(X[train_mask]), scaler.transform(X[test_mask])
            ridge = RidgeCV(alphas=RIDGE_ALPHAS_EXTENSION, cv=KFold(10, shuffle=True, random_state=202451),
                            scoring='neg_mean_squared_error').fit(X_train, y[train_mask])
            prediction = ridge.predict(X_test)
            residual = y[test_mask] - prediction
            result_rows.append({'year':year,'quarter':quarter,'investor_group':'all_s12',
                                'embedding_model':model,'dimension':dimension,'firms':len(panel),
                                'training_firms':int(train_mask.sum()),'test_firms':int(test_mask.sum()),
                                'selected_alpha':float(ridge.alpha_),
                                'test_variance_r2':float(1 - np.var(residual)/np.var(y[test_mask])),
                                'test_conventional_r2':float(r2_score(y[test_mask], prediction)),
                                'test_prediction_correlation':float(np.corrcoef(y[test_mask],prediction)[0,1])})
    quarter_results = pd.DataFrame(result_rows)
    quarter_fit = pd.DataFrame(fit_rows)
    save_parquet(quarter_results, result_path)
    save_parquet(quarter_fit, fit_path)
    all_s12_result_parts.append(quarter_results)
    all_s12_fit_parts.append(quarter_fit)
    print(f'{label}: all-S12 control complete ({shape[0]:,} funds)')

all_s12_results = pd.concat(all_s12_result_parts, ignore_index=True)
all_s12_fit_inventory = pd.concat(all_s12_fit_parts, ignore_index=True)
save_parquet(all_s12_results, ALL_S12_ROOT / 'eighty_quarter_results.parquet')
save_parquet(all_s12_fit_inventory, ALL_S12_ROOT / 'fit_inventory.parquet')
extension_results_with_all_s12 = pd.concat([behavioral_benchmark, all_s12_results], ignore_index=True)
extension_summary_with_all_s12 = (extension_results_with_all_s12.groupby(
    ['investor_group','embedding_model','dimension'], as_index=False)
    .agg(quarters=('quarter','size'), mean_test_variance_r2=('test_variance_r2','mean'),
         median_test_variance_r2=('test_variance_r2','median'), standard_deviation_r2=('test_variance_r2','std')))
save_parquet(extension_results_with_all_s12, ALL_S12_ROOT / 'extension_results_with_all_s12.parquet')
save_parquet(extension_summary_with_all_s12, ALL_S12_ROOT / 'extension_summary_with_all_s12.parquet')
display(extension_summary_with_all_s12.sort_values(['embedding_model','dimension','mean_test_variance_r2'], ascending=[True,True,False]))

## 22. GFC and COVID-19 performance

This section tests whether the relative effectiveness of active and passive holdings changes during market stress. The windows are fixed before inspecting the results:

- **Global Financial Crisis:** 2007Q3–2009Q2, covering the credit-market disruption through the recession trough.
- **Acute COVID-19 disruption:** 2020Q1–2020Q4, covering the crash, emergency policy response, and initial recovery.
- **Other quarters:** the remaining 67 evaluation quarters.

The COVID window contains only four observations and the GFC window eight, so crisis means are descriptive rather than standalone significance tests. Formal inference will use quarter-paired differences and block resampling. The primary tables compare only S12 groups; combined S12–S34 is reported separately as the original-replication reference.

In [ ]:
crisis_results = extension_results_with_all_s12.copy()
crisis_results['quarter_period'] = pd.PeriodIndex(
    year=crisis_results['year'], quarter=crisis_results['quarter'], freq='Q'
)
gfc = crisis_results['quarter_period'].between(pd.Period('2007Q3'), pd.Period('2009Q2'))
covid = crisis_results['quarter_period'].between(pd.Period('2020Q1'), pd.Period('2020Q4'))
crisis_results['market_regime'] = np.select(
    [gfc, covid], ['GFC_2007Q3_2009Q2','COVID_2020Q1_2020Q4'], default='Other_quarters'
)
s12_groups = ['all_s12','passive_like','stable_stock_picker','benchmark_rebalancer','tactical_active']
s12_crisis_results = crisis_results[crisis_results['investor_group'].isin(s12_groups)].copy()

crisis_summary = (s12_crisis_results.groupby(
    ['market_regime','investor_group','embedding_model','dimension'], as_index=False)
    .agg(quarters=('quarter','size'), mean_test_variance_r2=('test_variance_r2','mean'),
         median_test_variance_r2=('test_variance_r2','median'),
         minimum_test_variance_r2=('test_variance_r2','min'),
         maximum_test_variance_r2=('test_variance_r2','max')))

all_s12_regime = crisis_summary[crisis_summary['investor_group'].eq('all_s12')][
    ['market_regime','embedding_model','dimension','mean_test_variance_r2']
].rename(columns={'mean_test_variance_r2':'all_s12_mean_r2'})
crisis_summary = crisis_summary.merge(
    all_s12_regime, on=['market_regime','embedding_model','dimension'], how='left', validate='many_to_one'
)
crisis_summary['difference_from_all_s12'] = crisis_summary['mean_test_variance_r2'] - crisis_summary['all_s12_mean_r2']

passive_regime = crisis_summary[crisis_summary['investor_group'].eq('passive_like')][
    ['market_regime','embedding_model','dimension','mean_test_variance_r2']
].rename(columns={'mean_test_variance_r2':'passive_like_mean_r2'})
crisis_summary = crisis_summary.merge(
    passive_regime, on=['market_regime','embedding_model','dimension'], how='left', validate='many_to_one'
)
crisis_summary['difference_from_passive_like'] = crisis_summary['mean_test_variance_r2'] - crisis_summary['passive_like_mean_r2']

CRISIS_ROOT = EXTENSION_ROOT / 'benchmarks' / 'crisis_performance'
save_parquet(crisis_results, CRISIS_ROOT / 'quarterly_results_with_regimes.parquet')
save_parquet(crisis_summary, CRISIS_ROOT / 's12_crisis_summary.parquet')

for regime in ['GFC_2007Q3_2009Q2','COVID_2020Q1_2020Q4','Other_quarters']:
    print(f'\n{regime}')
    display(crisis_summary[crisis_summary['market_regime'].eq(regime)].sort_values(
        ['embedding_model','dimension','mean_test_variance_r2'], ascending=[True,True,False]
    )[['investor_group','embedding_model','dimension','quarters','mean_test_variance_r2',
       'difference_from_all_s12','difference_from_passive_like']])

crisis_model_ranking = (crisis_summary.assign(
    specification=lambda x: x['investor_group'] + '__' + x['embedding_model'] + '__' + x['dimension'].astype(str) + 'D')
    .sort_values(['market_regime','mean_test_variance_r2'], ascending=[True,False])
    .groupby('market_regime', as_index=False).head(10))
display(crisis_model_ranking[['market_regime','specification','quarters','mean_test_variance_r2']])

## 23. Equal-fund-count RS-Binary robustness

The active-fund advantage appears in RS-Binary, so this robustness test equalizes the number of funds across the four behavioral groups within every quarter. Five deterministic random subsamples are fitted. The common firm universe, target, train/test assignment, scaling rule, and ridge procedure remain unchanged. Embeddings and results are checkpointed for every seed.

In [ ]:
MATCHED_ROOT = EXTENSION_ROOT / 'benchmarks' / 'equal_fund_count_binary'
MATCHED_SEEDS = [101, 202, 303, 404, 505]
matched_result_parts = []
matched_audit_parts = []
for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    firm_mapping = pd.read_parquet(GROUP_MATRIX_ROOT / label / 'firm_mapping.parquet')
    group_inputs = {}
    for group in embedding_groups:
        matrix_root = GROUP_MATRIX_ROOT / label / group
        group_inputs[group] = (
            sparse.load_npz(matrix_root / 'binary.npz'),
            pd.read_parquet(matrix_root / 'investor_mapping.parquet'),
        )
    matched_fund_count = min(matrix.shape[0] for matrix, _ in group_inputs.values())

    panel = pd.read_parquet(REPLICATION_ROOT / 'benchmarks' / label / 'evaluation_sample_and_split.parquet')
    panel = panel.merge(firm_mapping[['permco']], on='permco', how='inner', validate='one_to_one').copy()
    panel['log_me_extension'] = np.log(panel['jkp_firm_market_equity'])
    panel['log_be_extension'] = np.log(panel['jkp_book_equity'])
    target_model = LinearRegression().fit(panel[['log_be_extension']], panel['log_me_extension'])
    y = (panel['log_me_extension'] - target_model.predict(panel[['log_be_extension']])).to_numpy()
    train_mask = panel['sample'].eq('train').to_numpy()
    test_mask = panel['sample'].eq('test').to_numpy()

    for matched_seed in MATCHED_SEEDS:
        seed_root = MATCHED_ROOT / label / f'seed={matched_seed}'
        result_path = seed_root / 'valuation_results.parquet'
        audit_path = seed_root / 'fit_audit.parquet'
        if result_path.exists() and audit_path.exists():
            matched_result_parts.append(pd.read_parquet(result_path))
            matched_audit_parts.append(pd.read_parquet(audit_path))
            continue
        result_rows = []
        audit_rows = []
        for group_index, group in enumerate(embedding_groups):
            matrix, investor_mapping = group_inputs[group]
            rng = np.random.default_rng(matched_seed + year * 100 + quarter * 10 + group_index)
            selected_rows = np.sort(rng.choice(matrix.shape[0], size=matched_fund_count, replace=False))
            matched_matrix = matrix[selected_rows].tocsr()
            dense = matched_matrix.toarray().astype('float32')
            pca = PCA(n_components=10, svd_solver='randomized', random_state=202451 + matched_seed).fit(dense)
            assets = (pca.components_.T * np.sqrt(dense.shape[1])).astype('float32')
            embeddings = firm_mapping[['permco','firm_name','firm_index']].copy()
            for component in range(10):
                embeddings[f'embedding_{component + 1:03d}'] = assets[:,component]
            save_parquet(embeddings, seed_root / group / 'asset_embeddings_10d.parquet')
            X_all = panel[['permco']].merge(embeddings, on='permco', validate='one_to_one')
            for dimension in [4,10]:
                columns_x = [f'embedding_{i:03d}' for i in range(1,dimension + 1)]
                X = X_all[columns_x].to_numpy(dtype='float64')
                scaler = StandardScaler().fit(X[train_mask])
                X_train, X_test = scaler.transform(X[train_mask]), scaler.transform(X[test_mask])
                ridge = RidgeCV(alphas=RIDGE_ALPHAS_EXTENSION,
                    cv=KFold(10, shuffle=True, random_state=202451),
                    scoring='neg_mean_squared_error').fit(X_train, y[train_mask])
                prediction = ridge.predict(X_test)
                residual = y[test_mask] - prediction
                result_rows.append({'year':year,'quarter':quarter,'matched_seed':matched_seed,
                    'investor_group':group,'embedding_model':'rs_binary','dimension':dimension,
                    'matched_funds':matched_fund_count,'firms':len(panel),
                    'selected_alpha':float(ridge.alpha_),
                    'test_variance_r2':float(1 - np.var(residual)/np.var(y[test_mask])),
                    'test_conventional_r2':float(r2_score(y[test_mask],prediction)),
                    'test_prediction_correlation':float(np.corrcoef(y[test_mask],prediction)[0,1])})
            selected_keys = investor_mapping.iloc[selected_rows]['investor_key']
            firm_degrees = np.diff(matched_matrix.tocsc().indptr)
            audit_rows.append({'year':year,'quarter':quarter,'matched_seed':matched_seed,
                'investor_group':group,'matched_funds':matched_fund_count,'firms':matched_matrix.shape[1],
                'represented_firms':int((firm_degrees > 0).sum()),
                'unrepresented_firms':int((firm_degrees == 0).sum()),
                'positions':matched_matrix.nnz,'selected_fund_keys':','.join(selected_keys.astype(str))})
            del dense
        quarter_results = pd.DataFrame(result_rows)
        quarter_audit = pd.DataFrame(audit_rows)
        save_parquet(quarter_results, result_path)
        save_parquet(quarter_audit, audit_path)
        matched_result_parts.append(quarter_results)
        matched_audit_parts.append(quarter_audit)
    print(f'{label}: five equal-count seeds complete at {matched_fund_count:,} funds per group')

matched_binary_results = pd.concat(matched_result_parts, ignore_index=True)
matched_binary_audit = pd.concat(matched_audit_parts, ignore_index=True)
matched_binary_summary = (matched_binary_results.groupby(['investor_group','dimension'], as_index=False)
    .agg(seed_quarters=('quarter','size'), mean_test_variance_r2=('test_variance_r2','mean'),
         standard_deviation_r2=('test_variance_r2','std'),
         minimum_matched_funds=('matched_funds','min'), median_matched_funds=('matched_funds','median')))
save_parquet(matched_binary_results, MATCHED_ROOT / 'all_results.parquet')
save_parquet(matched_binary_audit, MATCHED_ROOT / 'all_fit_audits.parquet')
save_parquet(matched_binary_summary, MATCHED_ROOT / 'summary.parquet')
display(matched_binary_summary.sort_values(['dimension','mean_test_variance_r2'], ascending=[True,False]))
display(matched_binary_audit.groupby('investor_group', as_index=False).agg(
    maximum_unrepresented_firms=('unrepresented_firms','max'),
    mean_unrepresented_firms=('unrepresented_firms','mean')))

## 24. Quarter-paired and crisis-window inference

For the full 79-quarter panel, confidence intervals use a circular moving-block bootstrap with four-quarter blocks, preserving short-run serial dependence. Comparisons are paired within quarter against passive-like funds and all S12. For the eight-quarter GFC and four-quarter COVID windows, exact sign-flip randomization p-values are reported; these remain low-power descriptive tests.

In [ ]:
from itertools import product

def circular_block_bootstrap_mean(values, block_length=4, repetitions=10000, seed=202451):
    values = np.asarray(values, dtype='float64')
    n = len(values)
    rng = np.random.default_rng(seed)
    estimates = np.empty(repetitions)
    blocks_needed = int(np.ceil(n / block_length))
    for repetition in range(repetitions):
        starts = rng.integers(0, n, size=blocks_needed)
        indices = np.concatenate([(start + np.arange(block_length)) % n for start in starts])[:n]
        estimates[repetition] = values[indices].mean()
    lower, upper = np.quantile(estimates, [0.025,0.975])
    p_value = 2 * min((estimates <= 0).mean(), (estimates >= 0).mean())
    return float(values.mean()), float(lower), float(upper), float(min(p_value,1.0))

def exact_sign_flip_pvalue(values):
    values = np.asarray(values, dtype='float64')
    observed = abs(values.mean())
    means = [abs(np.mean(values * np.asarray(signs))) for signs in product([-1,1], repeat=len(values))]
    return float(np.mean(np.asarray(means) >= observed - 1e-15))

inference_rows = []
analysis = extension_results_with_all_s12[extension_results_with_all_s12['investor_group'].isin(s12_groups)].copy()
for model in ['rs_binary','rs_l_min']:
    for dimension in [4,10]:
        spec = analysis[analysis['embedding_model'].eq(model) & analysis['dimension'].eq(dimension)]
        wide = spec.pivot(index=['year','quarter'], columns='investor_group', values='test_variance_r2').sort_index()
        for comparison_group in ['stable_stock_picker','benchmark_rebalancer','tactical_active']:
            for reference_group in ['passive_like','all_s12']:
                difference = (wide[comparison_group] - wide[reference_group]).dropna()
                mean, lower, upper, p_value = circular_block_bootstrap_mean(
                    difference.to_numpy(), seed=202451 + dimension + len(inference_rows)
                )
                inference_rows.append({'sample':'all_79_quarters','embedding_model':model,'dimension':dimension,
                    'comparison_group':comparison_group,'reference_group':reference_group,
                    'quarters':len(difference),'mean_r2_difference':mean,'ci_2_5':lower,'ci_97_5':upper,
                    'p_value':p_value,'inference_method':'circular_block_bootstrap_4q'})
                for regime, start, end in [('GFC','2007Q3','2009Q2'),('COVID','2020Q1','2020Q4')]:
                    periods = pd.PeriodIndex([f'{y}Q{q}' for y,q in difference.index], freq='Q')
                    mask = (periods >= pd.Period(start)) & (periods <= pd.Period(end))
                    event_difference = difference.to_numpy()[mask]
                    inference_rows.append({'sample':regime,'embedding_model':model,'dimension':dimension,
                        'comparison_group':comparison_group,'reference_group':reference_group,
                        'quarters':len(event_difference),'mean_r2_difference':float(event_difference.mean()),
                        'ci_2_5':np.nan,'ci_97_5':np.nan,'p_value':exact_sign_flip_pvalue(event_difference),
                        'inference_method':'exact_paired_sign_flip'})
inference_results = pd.DataFrame(inference_rows)
INFERENCE_ROOT = EXTENSION_ROOT / 'benchmarks' / 'statistical_inference'
save_parquet(inference_results, INFERENCE_ROOT / 'paired_inference.parquet')
display(inference_results.sort_values(['sample','embedding_model','dimension','reference_group','mean_r2_difference'],
                                      ascending=[True,True,True,True,False]))

## 25. Factorial Active Share and turnover effects

The equal-count design forms a 2×2 experiment: low/high Active Share crossed with low/high turnover. Results are first averaged across the five matched seeds within each quarter, ensuring that the inferential sample remains 79 quarters rather than treating repeated subsamples as independent observations. We estimate the marginal Active Share effect, marginal turnover effect, and their interaction, with four-quarter circular block-bootstrap confidence intervals.

In [ ]:
matched_quarter_means = (matched_binary_results.groupby(
    ['year','quarter','investor_group','dimension'], as_index=False)['test_variance_r2'].mean())
factorial_rows = []
factorial_quarter_rows = []
for dimension in [4,10]:
    wide = (matched_quarter_means[matched_quarter_means['dimension'].eq(dimension)]
            .pivot(index=['year','quarter'], columns='investor_group', values='test_variance_r2')
            .sort_index())
    contrasts = {
        'active_share_main_effect': 0.5 * ((wide['stable_stock_picker'] - wide['passive_like'])
                                           + (wide['tactical_active'] - wide['benchmark_rebalancer'])),
        'turnover_main_effect': 0.5 * ((wide['benchmark_rebalancer'] - wide['passive_like'])
                                      + (wide['tactical_active'] - wide['stable_stock_picker'])),
        'active_share_x_turnover': (wide['tactical_active'] - wide['stable_stock_picker']
                                      - wide['benchmark_rebalancer'] + wide['passive_like']),
    }
    for effect_index, (effect, values) in enumerate(contrasts.items()):
        mean, lower, upper, p_value = circular_block_bootstrap_mean(
            values.to_numpy(), block_length=4, repetitions=10000,
            seed=202451 + dimension * 100 + effect_index
        )
        factorial_rows.append({'dimension':dimension,'effect':effect,'quarters':len(values),
                               'mean_r2_effect':mean,'ci_2_5':lower,'ci_97_5':upper,
                               'p_value':p_value,'inference_method':'circular_block_bootstrap_4q'})
        for (year, quarter), value in values.items():
            factorial_quarter_rows.append({'year':year,'quarter':quarter,'dimension':dimension,
                                           'effect':effect,'r2_effect':value})
factorial_inference = pd.DataFrame(factorial_rows)
factorial_quarter_effects = pd.DataFrame(factorial_quarter_rows)
FACTORIAL_ROOT = EXTENSION_ROOT / 'benchmarks' / 'factorial_effects'
save_parquet(factorial_inference, FACTORIAL_ROOT / 'block_bootstrap_inference.parquet')
save_parquet(factorial_quarter_effects, FACTORIAL_ROOT / 'quarterly_effects.parquet')
display(factorial_inference)

## 26. Spectral variance versus economic information

For each quarter and investor group, RS-Binary is refitted at 20 components. We then record two distinct curves for dimensions 1–20:

1. **Cumulative holdings variance explained:** how well the components reconstruct variation in the investor–firm matrix.
2. **Out-of-sample valuation R²:** how economically useful the same cumulative components are.

The analysis includes the four behavioral groups and all eligible S12 funds. Ridge regularization is reselected at every dimension using the original training-only ten-fold procedure. Incremental predictive contribution is $R_k^2-R_{k-1}^2$; because this is out-of-sample and regularized, individual increments may be negative.

In [ ]:
SPECTRAL_ROOT = EXTENSION_ROOT / 'benchmarks' / 'binary_dimension_curves'
SPECTRAL_GROUPS = embedding_groups + ['all_s12']
spectral_result_parts = []
spectral_fit_parts = []
for year, quarter in QUARTERS[1:]:
    label = qlabel(year, quarter)
    result_path = SPECTRAL_ROOT / label / 'dimension_results.parquet'
    fit_path = SPECTRAL_ROOT / label / 'spectral_fit.parquet'
    if result_path.exists() and fit_path.exists():
        spectral_result_parts.append(pd.read_parquet(result_path))
        spectral_fit_parts.append(pd.read_parquet(fit_path))
        continue
    firm_mapping = pd.read_parquet(GROUP_MATRIX_ROOT / label / 'firm_mapping.parquet')
    group_matrices = {group:sparse.load_npz(GROUP_MATRIX_ROOT / label / group / 'binary.npz')
                      for group in embedding_groups}
    group_matrices['all_s12'] = sparse.vstack([group_matrices[group] for group in embedding_groups], format='csr')

    panel = pd.read_parquet(REPLICATION_ROOT / 'benchmarks' / label / 'evaluation_sample_and_split.parquet')
    panel = panel.merge(firm_mapping[['permco']], on='permco', how='inner', validate='one_to_one').copy()
    panel['log_me_extension'] = np.log(panel['jkp_firm_market_equity'])
    panel['log_be_extension'] = np.log(panel['jkp_book_equity'])
    target_model = LinearRegression().fit(panel[['log_be_extension']], panel['log_me_extension'])
    y = (panel['log_me_extension'] - target_model.predict(panel[['log_be_extension']])).to_numpy()
    train_mask = panel['sample'].eq('train').to_numpy()
    test_mask = panel['sample'].eq('test').to_numpy()
    quarter_results = []
    quarter_fits = []
    for group in SPECTRAL_GROUPS:
        dense = group_matrices[group].toarray().astype('float32')
        started = time.perf_counter()
        pca = PCA(n_components=20, svd_solver='randomized', random_state=202451).fit(dense)
        assets = (pca.components_.T * np.sqrt(dense.shape[1])).astype('float32')
        cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
        embeddings = firm_mapping[['permco','firm_name','firm_index']].copy()
        for component in range(20):
            embeddings[f'embedding_{component + 1:03d}'] = assets[:,component]
        save_parquet(embeddings, SPECTRAL_ROOT / label / group / 'asset_embeddings_20d.parquet')
        X_all = panel[['permco']].merge(embeddings, on='permco', validate='one_to_one')
        prior_r2 = 0.0
        for dimension in range(1,21):
            columns_x = [f'embedding_{i:03d}' for i in range(1,dimension + 1)]
            X = X_all[columns_x].to_numpy(dtype='float64')
            scaler = StandardScaler().fit(X[train_mask])
            X_train, X_test = scaler.transform(X[train_mask]), scaler.transform(X[test_mask])
            ridge = RidgeCV(alphas=RIDGE_ALPHAS_EXTENSION,
                cv=KFold(10, shuffle=True, random_state=202451),
                scoring='neg_mean_squared_error').fit(X_train, y[train_mask])
            prediction = ridge.predict(X_test)
            residual = y[test_mask] - prediction
            r2 = float(1 - np.var(residual)/np.var(y[test_mask]))
            quarter_results.append({'year':year,'quarter':quarter,'investor_group':group,
                'dimension':dimension,'test_variance_r2':r2,
                'incremental_test_variance_r2':r2-prior_r2,
                'cumulative_explained_variance':float(cumulative_variance[dimension-1]),
                'selected_alpha':float(ridge.alpha_),'firms':len(panel),
                'investors':dense.shape[0]})
            prior_r2 = r2
        quarter_fits.append({'year':year,'quarter':quarter,'investor_group':group,
            'investors':dense.shape[0],'firms':dense.shape[1],
            'fit_seconds':time.perf_counter()-started,
            'variance_explained_4d':float(cumulative_variance[3]),
            'variance_explained_10d':float(cumulative_variance[9]),
            'variance_explained_20d':float(cumulative_variance[19])})
        del dense
    quarter_results = pd.DataFrame(quarter_results)
    quarter_fits = pd.DataFrame(quarter_fits)
    save_parquet(quarter_results, result_path)
    save_parquet(quarter_fits, fit_path)
    spectral_result_parts.append(quarter_results)
    spectral_fit_parts.append(quarter_fits)
    print(f'{label}: Binary dimension curves complete')

binary_dimension_results = pd.concat(spectral_result_parts, ignore_index=True)
binary_spectral_fits = pd.concat(spectral_fit_parts, ignore_index=True)
binary_dimension_summary = (binary_dimension_results.groupby(['investor_group','dimension'], as_index=False)
    .agg(quarters=('quarter','size'), mean_test_variance_r2=('test_variance_r2','mean'),
         mean_incremental_r2=('incremental_test_variance_r2','mean'),
         mean_cumulative_variance=('cumulative_explained_variance','mean'),
         standard_deviation_r2=('test_variance_r2','std')))
save_parquet(binary_dimension_results, SPECTRAL_ROOT / 'all_quarter_results.parquet')
save_parquet(binary_spectral_fits, SPECTRAL_ROOT / 'all_spectral_fits.parquet')
save_parquet(binary_dimension_summary, SPECTRAL_ROOT / 'dimension_summary.parquet')
display(binary_dimension_summary[binary_dimension_summary['dimension'].isin([1,2,3,4,5,10,15,20])])